# HRSBF — Colab ↔ GitHub Worker\n\nEste notebook transforma a sessão do Colab em um worker do projeto. Ele lê a fila do GitHub, executa scripts Python do próprio repositório e envia status/logs de volta.\n\n**Antes da primeira execução:** no painel Secrets do Colab, crie `GITHUB_TOKEN` com acesso de leitura e escrita em Contents apenas para este repositório.

In [ ]:
from google.colab import userdata\nimport os, pathlib, subprocess\n\nREPO = 'https://github.com/contatofilipevono-rgb/hrsbf-bent-verification.git'\nROOT = pathlib.Path('/content/hrsbf-bent-verification')\ntoken = userdata.get('GITHUB_TOKEN')\nif not token:\n    raise RuntimeError('Crie o segredo GITHUB_TOKEN no painel Secrets do Colab.')\n\nsubprocess.run(['gh', 'auth', 'login', '--hostname', 'github.com', '--with-token'], input=token, text=True, check=True)\nsubprocess.run(['gh', 'auth', 'setup-git'], check=True)\n\nif not (ROOT / '.git').exists():\n    subprocess.run(['git', 'clone', REPO, str(ROOT)], check=True)\nelse:\n    subprocess.run(['git', '-C', str(ROOT), 'pull', '--rebase', 'origin', 'main'], check=True)\n\nos.chdir(ROOT)\nprint('GitHub conectado. Repositório pronto em', ROOT)\n

In [ ]:
# Diagnóstico da GPU\np = subprocess.run(['nvidia-smi', '-L'], text=True, capture_output=True)\nprint(p.stdout if p.returncode == 0 else 'GPU não detectada. Troque o runtime do Colab para GPU.')\n

## Teste de ponta a ponta\nEsta célula processa no máximo um job pendente. A fila inicial roda o verificador standalone e salva o resultado no GitHub.

In [ ]:
!python colab_worker.py --once\n

## Worker contínuo\nDepois do teste, rode esta célula. Enquanto a sessão Colab estiver viva, ela consulta o GitHub a cada 30 segundos e executa novos jobs.

In [ ]:
!python colab_worker.py --poll 30\n